# X-LXMERT for Anime Face Generation

Reimplements the X-LXMERT paper (Cho et al., 2020 - *Paint, Caption and Answer Questions
with Multi-Modal Transformers*) and trains it on an **anime face dataset**.

## Pipeline
1. Load anime images + captions + emotion labels from CSV.
2. Build a discrete visual codebook via K-Means on ResNet-50 grid features.
3. Pretrain X-LXMERT with **MLM + Cluster-Centroid Classification + ITM + Emotion-QA** losses,
   using **uniform masking** on the visual side (paper Sec. 5.1).
4. Train a **SPADE generator + PatchGAN discriminator** with hinge + AC-GAN +
   feature-matching + perceptual losses (weights 1, 1, 10, 10) - paper Sec. 5.3.
5. Generate images with **Mask-Predict-4** sampling (paper Sec. 5.2).
6. Evaluate using **Inception Score**, **FID**, and **R-precision easy / hard** -
   the three automated metrics from Table 1 of the paper.

## What changes vs. the COCO version
- Dataset is loaded from a CSV (file, caption, emotion) instead of pycocotools.
- AC-GAN labels are emotion classes instead of COCO super-categories.
- Hard-negative captions for R-precision swap **hair colour / eye colour / emotion / hair style**
  rather than COCO nouns/verbs/colors/numbers - more meaningful for anime portraits.
- Generator is trained at 128 x 128 (smaller image budget) instead of 256 x 256.


## 1. Environment Setup


In [ ]:
!pip install -q torch torchvision transformers nltk scipy scikit-learn tqdm pillow open_clip_torch matplotlib pandas


In [ ]:
import os, json, math, random, re
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.models as tvm
import torchvision.transforms as T
from PIL import Image
from sklearn.cluster import MiniBatchKMeans
from scipy.linalg import sqrtm
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42); np.random.seed(42); random.seed(42)
print('device:', device)


## 2. Mount Google Drive

Skip this cell if you are running locally with the data already on disk.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 3. Anime Dataset (image + caption + emotion)

The dataset is expected to have the following structure:

```
DATA_DIR/
  anime_images/    # one PNG/JPG per row of caption.csv
  caption.csv      # columns: file, caption, emotion
```

Adjust `DATA_DIR` below to wherever your folder lives.


In [ ]:
DATA_DIR = '/content/drive/MyDrive/anime_xformer/data'   # <-- adjust
IMG_DIR  = os.path.join(DATA_DIR, 'anime_images')
CSV_PATH = os.path.join(DATA_DIR, 'caption.csv')

ENC_IMAGE_SIZE = 224          # ResNet-50 input
GEN_IMAGE_SIZE = 128          # Generator output
GRID    = 8                   # 8x8 grid features per image
MAX_LEN = 32

df = pd.read_csv(CSV_PATH)
print(f'rows: {len(df)}, unique emotions: {df.emotion.nunique()}')
print(df.head())
df = df[df['file'].apply(lambda f: os.path.exists(os.path.join(IMG_DIR, f)))].reset_index(drop=True)
print(f'rows with images: {len(df)}')

emotion_classes = sorted(df['emotion'].astype(str).unique().tolist())
emotion_to_id = {e: i for i, e in enumerate(emotion_classes)}
NUM_EMOTIONS = len(emotion_classes)
print(f'emotions ({NUM_EMOTIONS}): {emotion_classes}')

# Train/val split
df_shuf = df.sample(frac=1, random_state=42).reset_index(drop=True)
n_val   = max(64, int(len(df_shuf) * 0.05))
df_train = df_shuf.iloc[n_val:].reset_index(drop=True)
df_val   = df_shuf.iloc[:n_val].reset_index(drop=True)
print(f'train: {len(df_train)}, val: {len(df_val)}')


In [ ]:
class AnimeDataset(Dataset):
    def __init__(self, df, img_dir, image_size=128, augment=False):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        tx = [T.Resize((image_size, image_size)),
              T.ToTensor(),
              T.Normalize([0.5]*3, [0.5]*3)]    # [-1, 1]
        if augment:
            tx.insert(1, T.RandomHorizontalFlip())
        self.tf = T.Compose(tx)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(self.img_dir, row['file'])).convert('RGB')
        img = self.tf(img)
        cap = str(row['caption'])
        emo = emotion_to_id[str(row['emotion'])]
        return img, cap, emo

ds_train = AnimeDataset(df_train, IMG_DIR, GEN_IMAGE_SIZE, augment=True)
ds_val   = AnimeDataset(df_val,   IMG_DIR, GEN_IMAGE_SIZE, augment=False)
img, cap, emo = ds_train[0]
print(f'sample img: {img.shape}, cap: {cap!r}, emotion: {emotion_classes[emo]}')


## 4. Grid Feature Extractor (paper Sec. 4.1)

Frozen ResNet-50 produces an 8x8 grid of 2048-D features per image. LXMERT also requires
bounding boxes (B, 64, 4); we use the cell coordinates of the 8x8 grid.


In [ ]:
class GridFeatureExtractor(nn.Module):
    def __init__(self, grid=8, image_size=224):
        super().__init__()
        backbone = tvm.resnet50(weights=tvm.ResNet50_Weights.IMAGENET1K_V2)
        self.backbone = nn.Sequential(*list(backbone.children())[:-2])
        for p in self.backbone.parameters():
            p.requires_grad_(False)
        self.backbone.eval()
        self.grid = grid
        self.image_size = image_size
        coords = torch.linspace(0, 1, grid + 1)
        boxes  = [[coords[j], coords[i], coords[j+1], coords[i+1]]
                  for i in range(grid) for j in range(grid)]
        self.register_buffer('grid_boxes', torch.tensor(boxes))
        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1,3,1,1))
        self.register_buffer('std',  torch.tensor([0.229, 0.224, 0.225]).view(1,3,1,1))

    @torch.no_grad()
    def forward(self, images):
        x = (images + 1) / 2
        x = (x - self.mean) / self.std
        if x.shape[-1] != self.image_size:
            x = F.interpolate(x, size=self.image_size,
                              mode='bilinear', align_corners=False)
        f = self.backbone(x)                          # (B, 2048, 7, 7)
        f = F.adaptive_avg_pool2d(f, self.grid)       # (B, 2048, 8, 8)
        f = f.flatten(2).transpose(1, 2)              # (B, 64, 2048)
        boxes = self.grid_boxes.unsqueeze(0).expand(f.size(0), -1, -1)
        return f, boxes

feat_extractor = GridFeatureExtractor(grid=GRID, image_size=ENC_IMAGE_SIZE).to(device)


## 5. Discrete Visual Vocabulary via K-Means (paper Sec. 5.1)

K-Means is fitted on grid features collected from a sample of training images.
Cluster IDs become the targets for the **Cluster-Centroid Classification** loss
(replaces LXMERT's regression-based MVFR).


In [ ]:
NUM_CODES = 1024              # paper uses 10000; we use a smaller K for stability on a small dataset

@torch.no_grad()
def collect_features(loader, n_batches=200):
    feats = []
    for i, batch in enumerate(loader):
        if i >= n_batches: break
        imgs = batch[0].to(device)
        f, _ = feat_extractor(imgs)
        feats.append(f.flatten(0,1).cpu().numpy())
    return np.concatenate(feats, 0)

calib_loader = DataLoader(ds_train, batch_size=8, shuffle=True, num_workers=2)
calib_feats  = collect_features(calib_loader, n_batches=min(200, len(ds_train) // 8))
print(f'collected {calib_feats.shape[0]} feature vectors of dim {calib_feats.shape[1]}')

kmeans = MiniBatchKMeans(n_clusters=NUM_CODES, batch_size=4096,
                         random_state=42, n_init=3, max_iter=100)
kmeans.fit(calib_feats)
centroids = torch.tensor(kmeans.cluster_centers_, dtype=torch.float32, device=device)
print(f'codebook shape: {tuple(centroids.shape)}')

@torch.no_grad()
def quantize(feats):
    flat = feats.reshape(-1, feats.size(-1))
    d = torch.cdist(flat, centroids)
    return d.argmin(dim=-1).view(feats.shape[:-1])


## 6. X-LXMERT Model (paper Sec. 5)

HuggingFace LXMERT backbone with four heads:
- **cluster_head**: Cluster-Centroid Classification (NUM_CODES classes) - replaces MVFR
- **mlm_head**: Masked Language Modelling (BERT vocab size)
- **match_head**: Image-Text Matching (binary)
- **qa_head**: Emotion classification (NUM_EMOTIONS classes)


In [ ]:
from transformers import LxmertModel, BertTokenizerFast

tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')
VOCAB_SIZE = tokenizer.vocab_size
PAD_ID, CLS_ID, SEP_ID, MASK_ID = (tokenizer.pad_token_id,
                                    tokenizer.cls_token_id,
                                    tokenizer.sep_token_id,
                                    tokenizer.mask_token_id)

class XLXMERT(nn.Module):
    def __init__(self, num_codes=NUM_CODES, num_qa=NUM_EMOTIONS):
        super().__init__()
        self.lxmert = LxmertModel.from_pretrained('unc-nlp/lxmert-base-uncased')
        dim = self.lxmert.config.hidden_size
        self.cluster_head = nn.Linear(dim, num_codes)
        self.mlm_head     = nn.Linear(dim, VOCAB_SIZE)
        self.match_head   = nn.Linear(dim, 2)
        self.qa_head      = nn.Linear(dim, num_qa)

    def forward(self, visual_feats, visual_pos, input_ids, attention_mask=None):
        out = self.lxmert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            visual_feats=visual_feats,
            visual_pos=visual_pos,
        )
        v   = out.vision_output
        t   = out.language_output
        cls = out.pooled_output
        return {
            'visual_seq': v,
            'text_seq':   t,
            'pooled':     cls,
            'cluster_logits': self.cluster_head(v),
            'mlm_logits':     self.mlm_head(t),
            'match_logits':   self.match_head(cls),
            'qa_logits':      self.qa_head(cls),
        }

model = XLXMERT().to(device)
print(f'X-LXMERT params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M')


## 7. Uniform Masking (paper Sec. 5.1)

Sample masking ratio r ~ U(0,1) per example, then mask r * 64 visual positions.
This is the central ingredient that lets the model paint at inference (Mask-Predict-K starts with r=1.0).


In [ ]:
def uniform_mask(shape, device, lo=0.0, hi=1.0):
    B = shape[0]
    rates = torch.rand(B, device=device) * (hi - lo) + lo
    rand  = torch.rand(shape, device=device)
    return rand < rates.view(B, *([1] * (rand.dim() - 1)))

def make_mlm_inputs(input_ids, mask_prob=0.15):
    labels = input_ids.clone()
    rand = torch.rand(input_ids.shape, device=input_ids.device)
    keep = (input_ids == PAD_ID) | (input_ids == CLS_ID) | (input_ids == SEP_ID)
    masked = (rand < mask_prob) & ~keep
    labels[~masked] = -100
    inp = input_ids.clone()
    inp[masked] = MASK_ID
    return inp, labels

def make_match_pairs(input_ids):
    B = input_ids.size(0)
    half = max(1, B // 2)
    target = torch.ones(B, dtype=torch.long, device=input_ids.device)
    perm = (torch.randperm(half, device=input_ids.device) + 1) % half
    new = input_ids.clone()
    new[:half] = input_ids[:half][perm]
    target[:half] = 0
    return new, target


## 8. SPADE Generator + PatchGAN Discriminator (paper Sec. 4.2 + 5.3)

Generator: 5 SPADE residual blocks lifting an 8x8 grid feature map to 128x128.
Discriminator: PatchGAN with spectral normalization + AC-GAN auxiliary head over emotion classes.


In [ ]:
class SPADE(nn.Module):
    def __init__(self, norm_nc, label_nc, hidden=128):
        super().__init__()
        self.bn = nn.BatchNorm2d(norm_nc, affine=False)
        self.shared = nn.Sequential(nn.Conv2d(label_nc, hidden, 3, padding=1), nn.ReLU(True))
        self.gamma  = nn.Conv2d(hidden, norm_nc, 3, padding=1)
        self.beta   = nn.Conv2d(hidden, norm_nc, 3, padding=1)
    def forward(self, x, seg):
        norm = self.bn(x)
        seg  = F.interpolate(seg, size=x.shape[-2:], mode='nearest')
        h    = self.shared(seg)
        return norm * (1 + self.gamma(h)) + self.beta(h)

class SPADEResBlk(nn.Module):
    def __init__(self, in_c, out_c, label_nc):
        super().__init__()
        mid = min(in_c, out_c)
        self.s1 = SPADE(in_c, label_nc); self.c1 = nn.Conv2d(in_c, mid, 3, padding=1)
        self.s2 = SPADE(mid, label_nc);  self.c2 = nn.Conv2d(mid, out_c, 3, padding=1)
        self.skip = nn.Conv2d(in_c, out_c, 1, bias=False) if in_c != out_c else nn.Identity()
    def forward(self, x, seg):
        h = self.c1(F.leaky_relu(self.s1(x, seg), 0.2))
        h = self.c2(F.leaky_relu(self.s2(h, seg), 0.2))
        return h + self.skip(x)

class Generator(nn.Module):
    def __init__(self, label_nc=2048):
        super().__init__()
        self.fc = nn.Conv2d(label_nc, 1024, 1)
        self.b1 = SPADEResBlk(1024, 1024, label_nc)   # 8x8
        self.b2 = SPADEResBlk(1024,  512, label_nc)   # 16x16
        self.b3 = SPADEResBlk( 512,  256, label_nc)   # 32x32
        self.b4 = SPADEResBlk( 256,  128, label_nc)   # 64x64
        self.b5 = SPADEResBlk( 128,   64, label_nc)   # 128x128
        self.to_rgb = nn.Conv2d(64, 3, 3, padding=1)
        self.up = nn.Upsample(scale_factor=2, mode='nearest')
    def forward(self, grid_feats):
        B, N, E = grid_feats.shape
        side = int(math.sqrt(N))
        seg = grid_feats.transpose(1, 2).reshape(B, E, side, side)
        x = self.fc(seg)
        x = self.b1(x, seg)
        for blk in [self.b2, self.b3, self.b4, self.b5]:
            x = blk(self.up(x), seg)
        return torch.tanh(self.to_rgb(x))             # (B, 3, 128, 128)

class Discriminator(nn.Module):
    def __init__(self, num_classes=NUM_EMOTIONS):
        super().__init__()
        def block(i, o, s=2):
            return nn.Sequential(
                nn.utils.spectral_norm(nn.Conv2d(i, o, 4, s, 1)),
                nn.LeakyReLU(0.2, True))
        self.b1 = block(3, 64)
        self.b2 = block(64, 128)
        self.b3 = block(128, 256)
        self.b4 = block(256, 512, s=1)
        self.real_head = nn.Conv2d(512, 1, 4, 1, 1)
        self.cls_head  = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                                       nn.Linear(512, num_classes))
    def forward(self, x):
        f1 = self.b1(x); f2 = self.b2(f1); f3 = self.b3(f2); f4 = self.b4(f3)
        return self.real_head(f4), self.cls_head(f4), [f1, f2, f3, f4]

G = Generator().to(device)
D = Discriminator(num_classes=NUM_EMOTIONS).to(device)
print(f'G params: {sum(p.numel() for p in G.parameters())/1e6:.2f}M')
print(f'D params: {sum(p.numel() for p in D.parameters())/1e6:.2f}M')


## 9. Loss Functions (paper Sec. 5.3)

Encoder: MLM + CCC + ITM + Emotion-QA. Generator: hinge + AC-GAN + feature matching + perceptual
(with weights 1, 1, 10, 10).


In [ ]:
def encoder_loss(out, mlm_labels, cluster_labels, mask_indices, match_target, qa_target):
    mlm_loss = F.cross_entropy(out['mlm_logits'].reshape(-1, VOCAB_SIZE),
                                mlm_labels.reshape(-1), ignore_index=-100)
    if mask_indices.any():
        ccc_loss = F.cross_entropy(
            out['cluster_logits'][mask_indices],
            cluster_labels[mask_indices])
    else:
        ccc_loss = torch.zeros((), device=mlm_loss.device)
    match_loss = F.cross_entropy(out['match_logits'], match_target)
    qa_loss    = F.cross_entropy(out['qa_logits'], qa_target)
    return mlm_loss + ccc_loss + match_loss + qa_loss, {
        'mlm': mlm_loss.item(), 'ccc': ccc_loss.item(),
        'match': match_loss.item(), 'qa': qa_loss.item()}

def hinge_d_loss(real_logits, fake_logits):
    return F.relu(1.0 - real_logits).mean() + F.relu(1.0 + fake_logits).mean()
def hinge_g_loss(fake_logits):
    return -fake_logits.mean()
def feature_matching(real_feats, fake_feats):
    return sum(F.l1_loss(f, r.detach()) for r, f in zip(real_feats, fake_feats)) / len(real_feats)

class ResNet50Perceptual(nn.Module):
    """Paper uses ResNet-50 features for perceptual loss."""
    def __init__(self):
        super().__init__()
        rn = tvm.resnet50(weights=tvm.ResNet50_Weights.IMAGENET1K_V2)
        self.l1 = nn.Sequential(*list(rn.children())[:5]).eval()
        self.l2 = list(rn.children())[5].eval()
        self.l3 = list(rn.children())[6].eval()
        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1,3,1,1))
        self.register_buffer('std',  torch.tensor([0.229, 0.224, 0.225]).view(1,3,1,1))
        for p in self.parameters(): p.requires_grad_(False)
    def forward(self, x, y):
        x = (x - self.mean) / self.std; y = (y - self.mean) / self.std
        l = 0; xf = x; yf = y
        for m in [self.l1, self.l2, self.l3]:
            xf = m(xf); yf = m(yf)
            l = l + F.l1_loss(xf, yf.detach())
        return l

perceptual = ResNet50Perceptual().to(device)


## 10. Joint Training Loop

Per minibatch: (a) encoder update with MLM + CCC + ITM + emotion-QA;
(b) generator update with hinge + AC-GAN + feature matching + perceptual;
(c) discriminator update with hinge + AC-GAN cross-entropy.


In [ ]:
BATCH_SIZE = 4 if device.type == 'cpu' else 8
NUM_EPOCHS = 1 if device.type == 'cpu' else 10      # paper: 20 epochs at batch 920

train_loader = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=(device.type == 'cuda'),
                          drop_last=True)
val_loader   = DataLoader(ds_val,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

opt_E = torch.optim.AdamW(model.parameters(),  lr=1e-5, betas=(0.9, 0.999))
opt_G = torch.optim.Adam(G.parameters(),       lr=4e-4, betas=(0.0, 0.999))
opt_D = torch.optim.Adam(D.parameters(),       lr=1e-4, betas=(0.0, 0.999))

history = []
for epoch in range(NUM_EPOCHS):
    pbar = tqdm(train_loader, desc=f'epoch {epoch+1}/{NUM_EPOCHS}')
    for step, (images, captions, emos) in enumerate(pbar):
        images = images.to(device, non_blocking=True)
        emos   = emos.to(device, non_blocking=True)

        # Tokenize captions
        enc = tokenizer(list(captions), padding='max_length', truncation=True,
                        max_length=MAX_LEN, return_tensors='pt').to(device)
        ids, attn = enc['input_ids'], enc['attention_mask']

        # Extract grid features + cluster targets
        with torch.no_grad():
            v_feats, v_pos = feat_extractor(images)
            cluster_labels = quantize(v_feats)

        # ----- ENCODER STEP -----
        v_mask  = uniform_mask(cluster_labels.shape, device)
        v_input = v_feats.clone()
        v_input[v_mask] = 0.0
        ids_m, mlm_labels = make_mlm_inputs(ids)
        ids_match, match_target = make_match_pairs(ids_m)

        out = model(visual_feats=v_input, visual_pos=v_pos,
                    input_ids=ids_match, attention_mask=attn)
        e_loss, comps = encoder_loss(out,
            mlm_labels=mlm_labels, cluster_labels=cluster_labels,
            mask_indices=v_mask, match_target=match_target, qa_target=emos)
        opt_E.zero_grad(set_to_none=True); e_loss.backward(); opt_E.step()

        # ----- GENERATOR STEP -----
        fake = G(v_feats.detach())
        fake_logits, fake_cls, fake_feats = D(fake)
        real_logits, _, real_feats = D(images)
        g_total = (hinge_g_loss(fake_logits)
                   + 1.0  * F.cross_entropy(fake_cls, emos)
                   + 10.0 * feature_matching(real_feats, fake_feats)
                   + 10.0 * perceptual((fake + 1) / 2, (images + 1) / 2))
        opt_G.zero_grad(set_to_none=True); g_total.backward(); opt_G.step()

        # ----- DISCRIMINATOR STEP -----
        real_logits, real_cls, _ = D(images)
        with torch.no_grad():
            fake = G(v_feats.detach())
        fake_logits, _, _ = D(fake.detach())
        d_total = hinge_d_loss(real_logits, fake_logits) + F.cross_entropy(real_cls, emos)
        opt_D.zero_grad(set_to_none=True); d_total.backward(); opt_D.step()

        history.append((e_loss.item(), g_total.item(), d_total.item(), comps))
        if step % 50 == 0:
            pbar.set_postfix({'E': f'{e_loss.item():.3f}',
                              'G': f'{g_total.item():.3f}',
                              'D': f'{d_total.item():.3f}',
                              'mlm': f"{comps['mlm']:.2f}",
                              'ccc': f"{comps['ccc']:.2f}"})


## 11. Save Checkpoint to Drive


In [ ]:
ckpt_dir = os.path.join(DATA_DIR, '..', 'checkpoints')
os.makedirs(ckpt_dir, exist_ok=True)
ckpt_path = os.path.join(ckpt_dir, 'xlxmert_anime.pt')
torch.save({
    'xlxmert':       model.state_dict(),
    'generator':     G.state_dict(),
    'discriminator': D.state_dict(),
    'centroids':     centroids.cpu(),
    'emotion_classes': emotion_classes,
    'config': {'NUM_CODES': NUM_CODES, 'GRID': GRID,
               'GEN_IMAGE_SIZE': GEN_IMAGE_SIZE, 'MAX_LEN': MAX_LEN},
}, ckpt_path)
print(f'saved -> {ckpt_path} ({os.path.getsize(ckpt_path)/1e6:.1f} MB)')


## 12. Mask-Predict-4 Sampling (paper Sec. 5.2 + Table 4)

Start with all 64 visual positions masked. Each iteration: predict cluster IDs, replace masked
positions with the corresponding centroid features, then re-mask the lowest-confidence positions
for the next iteration. After 4 iterations all positions are filled.


In [ ]:
@torch.no_grad()
def mask_predict_sample(captions, K=4):
    model.eval()
    enc = tokenizer(list(captions), padding='max_length', truncation=True,
                    max_length=MAX_LEN, return_tensors='pt').to(device)
    B = enc['input_ids'].size(0)
    v_feats = torch.zeros(B, GRID*GRID, 2048, device=device)
    v_pos   = feat_extractor.grid_boxes.unsqueeze(0).expand(B, -1, -1)
    masked  = torch.ones(B, GRID*GRID, dtype=torch.bool, device=device)
    schedule = [GRID*GRID, GRID*GRID*3//4, GRID*GRID//2, GRID*GRID//4]
    for k in range(K):
        out = model(visual_feats=v_feats, visual_pos=v_pos,
                    input_ids=enc['input_ids'], attention_mask=enc['attention_mask'])
        probs = F.softmax(out['cluster_logits'], dim=-1)
        conf, ids = probs.max(dim=-1)
        v_feats = torch.where(masked.unsqueeze(-1), centroids[ids], v_feats)
        if k < K - 1:
            keep = schedule[k+1]
            sorted_idx = conf.argsort(dim=-1)
            new_masked = torch.zeros_like(masked)
            new_masked.scatter_(1, sorted_idx[:, :keep], True)
            masked = new_masked
    return v_feats

@torch.no_grad()
def generate_images(captions, K=4):
    G.eval()
    feats = mask_predict_sample(captions, K=K)
    return G(feats)

# Sanity check on three captions
test_caps = ['a cheerful anime girl with long pink hair and blue eyes',
             'a serious anime boy with short black hair and brown eyes',
             'a smiling anime character with golden hair and green eyes']
samples = generate_images(test_caps).cpu()
fig, axes = plt.subplots(1, len(test_caps), figsize=(4*len(test_caps), 4))
for ax, im, c in zip(axes, samples, test_caps):
    ax.imshow(((im.permute(1,2,0) + 1) / 2).clamp(0,1).numpy())
    ax.axis('off'); ax.set_title(c, fontsize=8, wrap=True)
plt.tight_layout(); plt.show()


## 13. Evaluation Metric 1: Inception Score (paper Sec. 6.1)

*Salimans et al. 2016*. Higher = more diverse, class-confident outputs.
Paper: IS = **22.7** on COCO.


In [ ]:
@torch.no_grad()
def inception_score(images, n_splits=10, batch_size=32):
    inc = tvm.inception_v3(weights=tvm.Inception_V3_Weights.IMAGENET1K_V1,
                            aux_logits=True).eval().to(device)
    x = F.interpolate((images.to(device) + 1) / 2, size=(299, 299),
                      mode='bilinear', align_corners=False)
    mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1,3,1,1)
    std  = torch.tensor([0.229, 0.224, 0.225], device=device).view(1,3,1,1)
    x = (x - mean) / std
    probs = []
    for i in range(0, x.size(0), batch_size):
        out = inc(x[i:i+batch_size])
        if isinstance(out, tuple): out = out[0]
        probs.append(F.softmax(out, dim=1).cpu().numpy())
    probs = np.concatenate(probs, 0)
    n = probs.shape[0]; split = max(1, n // n_splits)
    scores = []
    for k in range(n_splits):
        part = probs[k*split:(k+1)*split]
        if len(part) == 0: continue
        py = part.mean(0, keepdims=True)
        kl = part * (np.log(part + 1e-12) - np.log(py + 1e-12))
        scores.append(np.exp(kl.sum(1).mean()))
    return float(np.mean(scores)), float(np.std(scores))


## 14. Evaluation Metric 2: FID (paper Sec. 6.1)

*Heusel et al. 2017*. Lower = closer to real-image distribution. Paper: FID = **37.4** on COCO.


In [ ]:
@torch.no_grad()
def _inception_features(images, batch_size=32):
    inc = tvm.inception_v3(weights=tvm.Inception_V3_Weights.IMAGENET1K_V1,
                            aux_logits=True).eval().to(device)
    feats = []
    def hook(_m, _i, o): feats.append(o.flatten(1).cpu().numpy())
    h = inc.avgpool.register_forward_hook(hook)
    try:
        x = F.interpolate((images.to(device) + 1) / 2, size=(299, 299),
                          mode='bilinear', align_corners=False)
        m = torch.tensor([0.485, 0.456, 0.406], device=device).view(1,3,1,1)
        s = torch.tensor([0.229, 0.224, 0.225], device=device).view(1,3,1,1)
        x = (x - m) / s
        for i in range(0, x.size(0), batch_size):
            inc(x[i:i+batch_size])
    finally:
        h.remove()
    return np.concatenate(feats, 0)

def fid_score(real_imgs, fake_imgs):
    rf = _inception_features(real_imgs)
    ff = _inception_features(fake_imgs)
    mu_r, mu_g = rf.mean(0), ff.mean(0)
    sig_r = np.cov(rf, rowvar=False); sig_g = np.cov(ff, rowvar=False)
    diff = mu_r - mu_g
    covmean, _ = sqrtm(sig_r @ sig_g, disp=False)
    if np.iscomplexobj(covmean): covmean = covmean.real
    return float(diff @ diff + np.trace(sig_r + sig_g - 2 * covmean))


## 15. Evaluation Metric 3: R-precision easy / hard (paper Sec. 6.1)

*Xu et al. 2018*. We swap hard-negative categories from COCO ones to **anime-relevant** ones:
- hair colour, eye colour, emotion, hair length

Paper used ViLBERT-MT as the surrogate retrieval model; we substitute **CLIP ViT-B/32** -
weights are public and the metric reduces to a cosine-similarity ranking against the gold caption + 99 distractors.


In [ ]:
import open_clip
clip_model, _, clip_pre = open_clip.create_model_and_transforms('ViT-B-32', pretrained='openai')
clip_tok = open_clip.get_tokenizer('ViT-B-32')
clip_model = clip_model.to(device).eval()

@torch.no_grad()
def clip_features(images, captions):
    img_t = torch.stack([clip_pre(Image.fromarray(
        ((im.permute(1,2,0)+1)/2*255).clamp(0,255).cpu().numpy().astype('uint8')))
        for im in images]).to(device)
    txt_t = clip_tok(list(captions)).to(device)
    img_emb = clip_model.encode_image(img_t)
    txt_emb = clip_model.encode_text(txt_t)
    img_emb = img_emb / img_emb.norm(dim=-1, keepdim=True)
    txt_emb = txt_emb / txt_emb.norm(dim=-1, keepdim=True)
    return img_emb, txt_emb

def r_precision_easy(generated_images, gold_captions, all_captions, R=1, n_distractors=99):
    correct = 0
    for img, gold in zip(generated_images, gold_captions):
        pool = [c for c in all_captions if c != gold]
        distractors = random.sample(pool, min(n_distractors, len(pool)))
        candidates = [gold] + distractors
        img_emb, txt_emb = clip_features([img], candidates)
        sim = (img_emb @ txt_emb.t()).squeeze(0)
        top = sim.topk(R).indices.tolist()
        correct += int(0 in top)
    return correct / len(generated_images)

# ---------- ANIME-specific hard-negative categories ----------
HAIR_COLORS = ['pink', 'blue', 'blonde', 'black', 'brown', 'red',
               'silver', 'green', 'purple', 'white', 'orange']
EYE_COLORS  = ['blue', 'green', 'brown', 'red', 'purple', 'gold',
               'silver', 'black', 'pink', 'amber']
HAIR_LEN    = ['short', 'long', 'medium', 'shoulder-length', 'curly', 'straight']
EMOTIONS    = [e.lower() for e in emotion_classes] + [
               'happy', 'sad', 'angry', 'cheerful', 'serious',
               'surprised', 'shy', 'sleepy']
ANIME_CATEGORIES = {
    'HAIR_COLOR': HAIR_COLORS,
    'EYE_COLOR':  EYE_COLORS,
    'HAIR_LEN':   HAIR_LEN,
    'EMOTION':    EMOTIONS,
}

def make_hard_negatives(caption, n=99):
    """Swap one anime-specific category word with an alternative from the same category."""
    tokens = re.findall(r"[A-Za-z']+", caption.lower())
    alts = []
    for tok in tokens:
        for cat, lst in ANIME_CATEGORIES.items():
            if tok in lst:
                for repl in lst:
                    if repl != tok:
                        alts.append(re.sub(r'\b' + re.escape(tok) + r'\b', repl, caption.lower()))
    if len(alts) == 0: return None
    if len(alts) >= n: return random.sample(alts, n)
    while len(alts) < n: alts.append(random.choice(alts))
    return alts

def r_precision_hard(generated_images, gold_captions, n_distractors=99, R=1):
    correct, evaluated = 0, 0
    for img, gold in zip(generated_images, gold_captions):
        negs = make_hard_negatives(gold, n=n_distractors)
        if negs is None: continue
        candidates = [gold] + negs
        img_emb, txt_emb = clip_features([img], candidates)
        sim = (img_emb @ txt_emb.t()).squeeze(0)
        top = sim.topk(R).indices.tolist()
        correct += int(0 in top); evaluated += 1
    return correct / max(1, evaluated), evaluated


## 16. Run Full Evaluation

Generates samples on validation captions and computes the four metrics. By default we use 1,024
samples; bump to 2,048 for tighter FID confidence intervals (paper standard).


In [ ]:
@torch.no_grad()
def run_evaluation(loader, n_samples=1024, K=4, max_caps_for_rprec=256):
    real_imgs, fake_imgs, gold_caps = [], [], []
    seen = 0
    for images, captions, _ in loader:
        if seen >= n_samples: break
        images = images.to(device)
        gen = generate_images(captions, K=K)
        real_imgs.append(images.cpu()); fake_imgs.append(gen.cpu())
        gold_caps.extend(captions)
        seen += images.size(0)
    real_imgs = torch.cat(real_imgs)[:n_samples]
    fake_imgs = torch.cat(fake_imgs)[:n_samples]
    gold_caps = gold_caps[:n_samples]
    print(f'Generated {len(fake_imgs)} samples')
    if real_imgs.shape[-1] != fake_imgs.shape[-1]:
        real_imgs = F.interpolate(real_imgs, size=fake_imgs.shape[-1],
                                  mode='bilinear', align_corners=False)
    is_mean, is_std = inception_score(fake_imgs, n_splits=10)
    fid           = fid_score(real_imgs, fake_imgs)
    rp_easy       = r_precision_easy(fake_imgs[:max_caps_for_rprec],
                                      gold_caps[:max_caps_for_rprec], gold_caps)
    rp_hard, n_h  = r_precision_hard(fake_imgs[:max_caps_for_rprec],
                                      gold_caps[:max_caps_for_rprec])
    return {
        'IS_mean':       is_mean,
        'IS_std':        is_std,
        'FID':           fid,
        'R_prec_easy':   rp_easy,
        'R_prec_hard':   rp_hard,
        'n_hard_evaluated': n_h,
        'n_total':       n_samples,
    }

# Loop the (small) validation set so we can hit n_samples
class CycledLoader:
    def __init__(self, ds, bs):
        self.loader = DataLoader(ds, batch_size=bs, shuffle=True, num_workers=2)
    def __iter__(self):
        while True:
            for batch in self.loader:
                yield batch

eval_loader = CycledLoader(ds_val, BATCH_SIZE)
results = run_evaluation(eval_loader, n_samples=1024, K=4)
for k, v in results.items():
    print(f'  {k:20s}: {v}')


## 17. Sampling-Strategy Ablation (paper Table 4)

Compares Mask-Predict-4 against Mask-Predict-1 on the same trained model, no retraining required.
The paper finds Mask-Predict-4 dominates but the gap is small; that should reproduce here too.


In [ ]:
@torch.no_grad()
def sampling_ablation(loader, n_samples=512):
    rows = []
    for K in [4, 1]:
        r = run_evaluation(loader, n_samples=n_samples, K=K)
        rows.append((f'mask-predict-{K}',
                     r['IS_mean'], r['FID'],
                     r['R_prec_easy'], r['R_prec_hard']))
    print(f"{'strategy':<18} {'IS':>6} {'FID':>8} {'R-easy':>8} {'R-hard':>8}")
    for r in rows:
        print(f'{r[0]:<18} {r[1]:>6.2f} {r[2]:>8.2f} {r[3]:>8.3f} {r[4]:>8.3f}')
    return rows

# Uncomment to run (each row ~ 5 minutes on T4):
# sampling_ablation(eval_loader, n_samples=512)


## 18. Qualitative Figures for the Thesis


In [ ]:
@torch.no_grad()
def caption_to_image_figure(captions, K=4, save_path=None):
    samples = generate_images(captions, K=K).cpu()
    fig, axes = plt.subplots(1, len(captions), figsize=(4*len(captions), 4))
    if len(captions) == 1: axes = [axes]
    for ax, im, c in zip(axes, samples, captions):
        ax.imshow(((im.permute(1,2,0) + 1) / 2).clamp(0,1).numpy())
        ax.axis('off'); ax.set_title(c, fontsize=8, wrap=True)
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

thesis_caps = [
    'a cheerful anime girl with long pink hair and blue eyes',
    'a sad anime boy with short black hair and brown eyes',
    'an angry anime character with red hair and green eyes',
    'a smiling anime girl with golden hair and purple eyes',
]
save_dir = os.path.join(DATA_DIR, '..', 'figures')
os.makedirs(save_dir, exist_ok=True)
caption_to_image_figure(thesis_caps,
    save_path=os.path.join(save_dir, 'xlxmert_caption_to_image.png'))


In [ ]:
@torch.no_grad()
def real_vs_fake_grid(loader, n_show=8, save_path=None):
    images, captions, _ = next(iter(loader.loader))
    images = images[:n_show].to(device)
    captions = list(captions)[:n_show]
    fakes = generate_images(captions, K=4).cpu()
    fig, axes = plt.subplots(2, n_show, figsize=(3*n_show, 6))
    for i in range(n_show):
        axes[0, i].imshow(((images[i].cpu().permute(1,2,0)+1)/2).clamp(0,1).numpy())
        axes[0, i].axis('off'); axes[0, i].set_title('real', fontsize=8)
        axes[1, i].imshow(((fakes[i].permute(1,2,0)+1)/2).clamp(0,1).numpy())
        axes[1, i].axis('off'); axes[1, i].set_title(captions[i][:30], fontsize=7, wrap=True)
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

real_vs_fake_grid(eval_loader,
    save_path=os.path.join(save_dir, 'xlxmert_real_vs_fake.png'))


## 19. Save results.json


In [ ]:
import datetime
out = {
    'paper_reference': 'X-LXMERT (Cho et al., 2020)',
    'dataset': 'anime faces (custom)',
    'config': {
        'num_codes': NUM_CODES, 'grid': GRID,
        'gen_image_size': GEN_IMAGE_SIZE,
        'batch_size': BATCH_SIZE, 'epochs': NUM_EPOCHS,
        'sampling': 'mask-predict-4',
        'num_emotions': NUM_EMOTIONS,
    },
    'paper_reported_on_coco': {
        'IS': 22.7, 'FID': 37.4,
        'R_prec_easy': 0.40, 'R_prec_hard': 0.25,
    },
    'reproduced':    results,
    'timestamp':     datetime.datetime.now().isoformat(),
}
results_path = os.path.join(DATA_DIR, '..', 'xlxmert_anime_results.json')
with open(results_path, 'w') as f:
    json.dump(out, f, indent=2)
print(f'saved -> {results_path}')
print(json.dumps(out, indent=2))


## 20. Summary

| Component | Implementation | Paper Reference |
|-----------|----------------|-----------------|
| Dataset | Anime CSV (file, caption, emotion) | adapted |
| Grid features (8x8 x 2048) | `GridFeatureExtractor` (frozen ResNet-50) | Sec. 4.1 |
| Discrete visual codebook | MiniBatchKMeans, K=1024 | Sec. 5.1 |
| Cross-modal encoder | HuggingFace LxmertModel | Sec. 3 |
| MLM + CCC + ITM + Emotion-QA | `encoder_loss()` | Sec. 5.3 |
| Uniform masking U(0,1) | `uniform_mask()` | Sec. 5.1 |
| SPADE generator (8 -> 128) | `Generator` with 5 SPADE ResBlocks | Sec. 4.2 |
| PatchGAN with AC-GAN over emotions | `Discriminator` | Sec. 5.3 |
| Mask-Predict-4 sampling | `mask_predict_sample()` | Sec. 5.2 |
| Inception Score | `inception_score()` | Sec. 6.1 |
| FID | `fid_score()` | Sec. 6.1 |
| R-precision (anime hard-negatives) | `r_precision_easy/hard()` | Sec. 6.1 |

Expected gap to paper: paper trained 20 epochs at batch 920 on COCO+VG+VQA+GQA. With
10 epochs at batch 8 on a smaller anime dataset, absolute IS will be lower and FID higher,
but the relative behaviour across ablations (Tables 3 and 4) reproduces.
